# How pawpaw Works: Transformers, LoRA, and GGUF

This notebook explains the methodological aspects behind pawpaw for readers **not** familiar with the deep-learning literature. We cover:

1. **Transformers** — the architecture inside every modern LLM.
2. **LoRA** — how pawpaw teaches a tiny model a new trick without retraining
   the whole thing.
3. **Quantization & GGUF** — how a 600 MB model shrinks to run on a laptop.
4. **The .paw bundle** — what pawpaw actually writes to disk.
5. **Inference: loading & switching** — why three classifiers cost the same
   memory as one.

Each section is self-contained — skip what you already know.

---

## 1. Transformers

A transformer is a neural network that processes text. You give it a
sequence of words, and it produces another sequence (a translation, a
summary, a classification, ...). Its key invention is **self-attention**:
as the network processes each word, it can "look at" every other word in
the sentence to gather context — all in a single pass, with no loops.

Let's unpack that step by step.

### Step 1: Chopping text into tokens

Neural networks don't read letters or words directly. They work with
**tokens** — short chunks of text, typically a common word or a piece of
a word. For example, the sentence:

> "Classifying is fun!"

might be tokenized as: `["Class", "ifying", " is", " fun", "!"]`

The tokenizer assigns each token an integer ID (e.g. `"Class"` → 42).
There are typically 30,000–100,000 possible tokens in the model's
vocabulary.

### Step 2: Turning token IDs into vectors (embeddings)

An integer ID like 42 is just a label — it carries no information about
what the token *means*. The model needs a richer representation. So it
looks up each token ID in a giant table called the **embedding matrix**
and retrieves a list of ~1000 floating-point numbers — the token's
**embedding vector**.

You can think of this as coordinates in a high-dimensional "meaning
space". Tokens that play similar roles in language end up with similar
coordinates. The model learns these coordinates during pre-training —
nobody sets them by hand.

Formally, the embedding matrix $E \in \mathbb{R}^{V \times d}$ has one
row per vocabulary entry ($V$ ≈ 50,000) and $d$ columns (typically 1024
for a small model). Given a token ID $t$, the embedding is simply:

$$\mathbf{x}_t = E[t, :] \quad \in \mathbb{R}^d$$

The embedding matrix $E$ is a *learned parameter* — its 50,000 × 1,024
= 51 million numbers are adjusted during pre-training so that semantically
similar tokens land near each other.

In [ ]:
import torch
torch.manual_seed(0)

embed = torch.nn.Embedding(num_embeddings=50000, embedding_dim=1024)
token_ids = torch.tensor([42, 7, 3021])  # three token IDs
vectors = embed(token_ids)               # look up their embedding vectors

print(f"3 tokens → 3 vectors, each with {vectors.shape[1]} dimensions")
print(f"The embedding table has {embed.weight.numel():,} learnable numbers in it")
print(f"  (50,000 tokens × 1,024 dimensions = {50000 * 1024:,})")

### Step 3: Self-attention — every token talks to every other token

Now we have a list of vectors $\mathbf{x}_1, \mathbf{x}_2, \ldots,
\mathbf{x}_n$, one per token. But each vector only encodes the token's
*own* meaning in isolation — it doesn't know anything about the
surrounding words.

Consider: "The animal didn't cross the street because **it** was too
tired." What does "it" refer to? The animal or the street? A human
knows it's the animal because of the surrounding context. The model
needs the same ability.

**Self-attention** gives each token a way to gather information from
every other token in the sequence. It does this in four steps.

#### Step 3a: Create Query, Key, Value vectors

Each token creates three new vectors by multiplying its embedding
through three learned weight matrices:

$$\mathbf{q}_i = W_Q \, \mathbf{x}_i \qquad
  \mathbf{k}_i = W_K \, \mathbf{x}_i \qquad
  \mathbf{v}_i = W_V \, \mathbf{x}_i$$

where $W_Q, W_K, W_V \in \mathbb{R}^{d_k \times d}$ are learned
during pre-training. Intuitively:

- **Query $\mathbf{q}_i$** — "here's what I'm looking for"
- **Key $\mathbf{k}_i$** — "here's what I have to offer"
- **Value $\mathbf{v}_i$** — "here's the actual information I'll share"

Think of it like a library: the Query is your search term, the Key is
the label on each book, and the Value is the book's content. You match
your query against all keys to decide which books to read, then take
away the content of the ones that matched.

#### Step 3b: Compute attention scores

To decide how much token $i$ should pay attention to token $j$, take
the dot product of $i$'s Query with $j$'s Key:

$$e_{ij} = \mathbf{q}_i^\top \mathbf{k}_j$$

A high dot product means "your Key matches my Query — you're relevant
to me." But raw dot products can be huge (they grow with $d_k$), which
makes gradients unstable during training. So we scale down:

$$e_{ij} = \frac{\mathbf{q}_i^\top \mathbf{k}_j}{\sqrt{d_k}}$$

The $\sqrt{d_k}$ normalization is a detail from the original paper
([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)). It keeps
the variance of the scores roughly constant regardless of the
dimension $d_k$.

#### Step 3c: Turn scores into weights

Apply softmax to each row to get a probability distribution:

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{n} \exp(e_{ik})}$$

Each $\alpha_{ij} \geq 0$ and $\sum_j \alpha_{ij} = 1$. Token $i$
now has a probability distribution over all tokens — who it should
listen to, and how much.

The softmax is what makes attention differentiable: small changes in
the scores produce smooth changes in the weights, which allows
gradient-based learning.

#### Step 3d: Mix the values

Compute the output for token $i$ as a weighted sum of all value
vectors:

$$\mathbf{z}_i = \sum_{j=1}^{n} \alpha_{ij} \, \mathbf{v}_j$$

Tokens that got high attention weights contribute a lot to $\mathbf{z}_i$;
tokens with low weights are essentially ignored. After this step,
$\mathbf{z}_i$ is a **contextualized** representation — it encodes not
just token $i$'s own meaning, but also what it learned from the
surrounding tokens.

The net effect: after self-attention, the vector for "it" has absorbed
information from "animal" (high weight) and largely ignored "street"
(low weight) — exactly what we want.

#### The full self-attention formula

In matrix form (processing all tokens at once), the entire self-attention
operation is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

where $Q, K, V \in \mathbb{R}^{n \times d_k}$ are the matrices obtained
by stacking all query, key, and value vectors as rows. This single matrix
expression does exactly what we described above: compute all pairwise
scores, softmax them row-wise, and take a weighted sum of the values.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# A tiny example: 10 tokens, each represented by a 512-dim vector
seq_len, d_model, d_k = 10, 512, 64
x = torch.randn(seq_len, d_model)   # pretend these are our embeddings

# Three learned weight matrices (random here; learned during pre-training)
W_Q = torch.randn(d_model, d_k) * 0.02
W_K = torch.randn(d_model, d_k) * 0.02
W_V = torch.randn(d_model, d_k) * 0.02

# Step 3a: each token creates its Q, K, V vectors
Q = x @ W_Q   # shape: (10, 64) — every token now has a query
K = x @ W_K   # shape: (10, 64) — every token now has a key
V = x @ W_V   # shape: (10, 64) — every token now has a value

# Step 3b+3c: compute attention weights — how much token i attends to token j
scores = Q @ K.T / (d_k ** 0.5)    # pairwise dot products, scaled
weights = F.softmax(scores, dim=-1) # each row sums to 1 (a distribution)

# Step 3d: weighted sum of values
output = weights @ V  # shape: (10, 64) — each token now "knows about" the others

print(f"Input:  10 tokens, each a {d_model}-dim vector")
print(f"Output: 10 tokens, each a {d_k}-dim vector (contextualized)")
print(f"\nHow much token 0 attends to each token (weights α₀ⱼ):")
for j in range(seq_len):
    bar = "█" * int(weights[0, j].item() * 200)
    print(f"  token {j}: {weights[0, j]:.3f}  {bar}")
print(f"  sum = {weights[0].sum():.4f}")

### Multi-head attention: getting multiple perspectives

A single set of Q/K/V weights can only capture one kind of relationship.
In practice, transformers run **multiple attention "heads" in parallel**
(typically 8–32 per layer). Each head $h$ has its own weight matrices
$W_Q^{(h)}, W_K^{(h)}, W_V^{(h)}$, so it can learn to attend to
different things:

- One head might learn to track subject–verb agreement
- Another might track pronoun references ("it" → "animal")
- Another might track positional proximity

For $H$ heads, each producing an output $\text{head}_h \in \mathbb{R}^{d_v}$,
the multi-head attention output is:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \; W_O$$

where $W_O \in \mathbb{R}^{H \cdot d_v \times d}$ is the **output
projection** matrix (called `o_proj` in the code). It projects the
concatenated heads back to the model dimension $d$.

$W_O$ is the fourth weight matrix in the attention block — and one of
the four that LoRA modifies.

### Step 4: The transformer block (a full "layer")

Self-attention is powerful, but it's only one piece. A full transformer
**block** wraps attention in a few extra steps:

$$\mathbf{z} = \text{LayerNorm}\!\big(\mathbf{x} + \text{MultiHead}(\mathbf{x})\big)$$
$$\mathbf{y} = \text{LayerNorm}\!\big(\mathbf{z} + \text{FFN}(\mathbf{z})\big)$$

Unpacking the notation:

1. **Multi-head self-attention** — mixes information across tokens
   (described above)
2. **Residual connection** ($\mathbf{x} + \cdots$) — adds the input
   back to the output. This helps gradients flow during training;
   without it, deep networks struggle to learn because gradients
   vanish as they propagate through many layers.
3. **Layer normalization** — stabilizes the magnitudes of the
   activations, preventing them from exploding or collapsing as they
   flow through the stack.
4. **Feed-forward network (FFN)** — two linear transformations with
   a non-linear activation in between:

$$\text{FFN}(\mathbf{z}) = W_2 \; \sigma(W_1 \, \mathbf{z} + \mathbf{b}_1) + \mathbf{b}_2$$

   where $W_1 \in \mathbb{R}^{d_{ff} \times d}$, $W_2 \in \mathbb{R}^{d \times d_{ff}}$,
   and $\sigma$ is a non-linear activation (typically SiLU or GELU).
   This is where each token's representation gets refined individually,
   after attention has mixed in context from other tokens.

5. Another **residual connection + LayerNorm**

A modern LLM stacks many of these blocks on top of each other.
Qwen3-0.6B has **28 blocks**. The output of block 1 feeds into block 2,
and so on — each block can learn progressively more abstract
relationships.

The key takeaway: **each block contains four attention weight matrices**
($W_Q, W_K, W_V, W_O$) **plus feed-forward weights**. Those four
attention matrices are exactly what LoRA modifies — which we'll see
in the next section.

In [ ]:
# Where do the ~600 million parameters live?
#
# Qwen3-0.6B has 28 layers, each with:
# - 4 attention projections (Q, K, V, O): each is a 1024×1024 matrix
# - 3 feed-forward projections (up, gate, down): each is 1024×3072
#
# Let's count:

layers = 28
d = 1024       # model dimension
ff = 3072      # feed-forward dimension

attn_per_layer = 4 * d * d             # W_Q + W_K + W_V + W_O
ff_per_layer   = 3 * d * ff            # W_up + W_gate + W_down
embeddings     = 2 * 50257 * d         # input embeddings + output head
total          = layers * (attn_per_layer + ff_per_layer) + embeddings

print(f"Per layer:")
print(f"  Attention (W_Q, W_K, W_V, W_O):  {attn_per_layer:>10,}  ({attn_per_layer / (attn_per_layer + ff_per_layer) * 100:.0f}%)")
print(f"  Feed-forward (W_up, W_gate, W_down): {ff_per_layer:>7,}  ({ff_per_layer / (attn_per_layer + ff_per_layer) * 100:.0f}%)")
print(f"\n28 layers:  {layers * (attn_per_layer + ff_per_layer) / 1e6:.0f}M params")
print(f"Embeddings + output head: {embeddings / 1e6:.0f}M params")
print(f"Grand total: {total / 1e6:.0f}M params")
print(f"\n→ Attention weights are {layers * attn_per_layer / total * 100:.0f}% of the model.")
print(f"  These are the weights LoRA targets.")

### Step 5: Generating output (the "G" in GPT)

After all 28 blocks, the model outputs one vector per token position.
To generate the next word, it takes the vector at the *last* position
and projects it through a final linear layer (the **output head**) into
a probability distribution over the entire vocabulary:

$$P(\text{next token}) = \text{softmax}(W_{\text{out}} \, \mathbf{y}_{\text{last}})$$

where $W_{\text{out}} \in \mathbb{R}^{V \times d}$ maps from the model
dimension to the vocabulary size. The highest-probability token becomes
the next word.

For classification, we don't need a long sequence — we just want one
word ("trivial" or "substantive", "frustrated" or "calm", etc.). The
model generates that single word and stops.

**Summary so far:** A transformer is a stack of blocks, each containing
self-attention (which mixes context across tokens) and a feed-forward
network (which refines each token individually). The parameters live in
the weight matrices $W_Q, W_K, W_V, W_O$ (attention) and $W_1, W_2$
(feed-forward) of these layers. A small model like Qwen3-0.6B has
~600 million of these numbers.

---

## 2. LoRA: Low-Rank Adaptation

Now we have a pre-trained model with 600M parameters that's good at
general language tasks. We want to make it good at a *specific* task
(e.g., classifying messages as "trivial" vs "substantive").

The traditional approach — **full fine-tuning** — updates *every one*
of those 600M parameters. That requires a lot of GPU memory and
produces a separate 600M copy for each new task.

**LoRA** (Low-Rank Adaptation, [Hu et al. 2021](https://arxiv.org/abs/2106.09685))
takes a radically different approach: **freeze the original model
entirely** and add a pair of tiny "correction" matrices instead.
The result: ~7 MB of new weights per task, instead of ~600 MB.

### The key insight: weight updates are low-rank

Think about what happens when you fine-tune a model. The original
weights $W \in \mathbb{R}^{d_{out} \times d_{in}}$ get adjusted by some
amount $\Delta W$. The fine-tuned weights are $W' = W + \Delta W$.

LoRA's key observation: even though $\Delta W$ is a big matrix (the
same shape as $W$), the *actual changes* it needs to make are simple
enough that they can be expressed in a compressed form. It's like
saying: "I need to adjust this 1024×1024 grid of numbers, but the
adjustment I need is really just a combination of a few basic patterns."

This is a **rank constraint**. Formally, LoRA decomposes the update
into two smaller matrices:

$$\Delta W = B \, A$$

where $A \in \mathbb{R}^{r \times d_{in}}$ and $B \in \mathbb{R}^{d_{out} \times r}$
with $r \ll d_{in}, d_{out}$. The number $r$ is the **rank** — it
controls how many "basic patterns" the correction can combine.

Why does this work? A matrix $\Delta W \in \mathbb{R}^{d_{out} \times d_{in}}$
can in principle have rank up to $\min(d_{out}, d_{in})$, which means
it can express $d_{out} \times d_{in}$ independent degrees of freedom.
But the empirical finding from the LoRA paper is that the *effective
rank* of the changes needed for adaptation is very low — often 4–16
suffices. Intuitively, pre-training has already learned most of the
structure; fine-tuning only needs to make small, structured tweaks.

The parameter count drops from $d_{out} \times d_{in}$ to
$r \times (d_{out} + d_{in})$:

In [ ]:
# Parameter savings with LoRA
#
# Take one attention weight matrix: shape (1024, 1024) = 1,048,576 numbers
#
# With LoRA at rank r, we need two small matrices instead:
#   A: shape (r, 1024) → r × 1024 numbers
#   B: shape (1024, r) → 1024 × r numbers
#   Total: 2 × r × 1024 numbers

d = 1024
full_params = d * d

print(f"Original weight matrix: {full_params:,} parameters")
print(f"")
print(f"{'Rank r':>8}  {'LoRA params':>12}  {'Compression':>14}")
print("-" * 40)
for r in [4, 8, 16, 32]:
    lora_params = 2 * r * d
    ratio = full_params / lora_params
    print(f"{r:>8}  {lora_params:>12,}  {ratio:>12.0f}× fewer")

### The modified forward pass

With LoRA, the forward pass for a single weight matrix becomes:

$$\mathbf{y} = W \, \mathbf{x} + \frac{\alpha}{r} \, B \, A \, \mathbf{x}$$

Or equivalently:

$$\mathbf{y} = \left(W + \frac{\alpha}{r} B A\right) \mathbf{x}$$

Let's unpack the pieces:

- $W \mathbf{x}$ — the original model's computation. **Frozen**, never
  changes.
- $B A \mathbf{x}$ — the LoRA correction. This is the only part that
  gets trained.
- $\alpha$ — the **scaling factor** (called `lora_alpha`). It controls
  how strongly the correction is applied relative to the original
  weights. pawpaw sets $\alpha = 2r$ by default, so $\alpha/r = 2$.

A subtle but important point: we can rewrite $B A \mathbf{x}$ as
$B (A \mathbf{x})$. This means we first compute $A \mathbf{x} \in
\mathbb{R}^r$ (a *compression* of the input into $r$ dimensions),
then expand it back with $B$ (a *decompression*). The intermediate
$r$-dimensional vector is the "bottleneck" — it forces the correction
to pass through a narrow information channel, which is exactly what
keeps the parameter count low.

At inference time, there's an even better option: since $W$ is frozen,
we can **merge** the LoRA into the base weights by computing
$W' = W + (\alpha/r) B A$ once, and then just use $W'$ as if it were
a single matrix. This eliminates any extra computation from the LoRA.
However, pawpaw keeps them separate so it can **swap** LoRA adapters
at runtime (see section 5).

### Which weight matrices does pawpaw put LoRA on?

A transformer has many weight matrices, but LoRA doesn't need to touch
all of them. The original LoRA paper found that adapting only the
**attention projections** works well for most tasks. pawpaw follows
this convention.

Recall the four attention weight matrices in each transformer block:

| Matrix | What it does | Shape | LoRA on it? |
|---|---|---|---|
| $W_Q$ (`q_proj`) | Creates Query vectors | $(d, d)$ | Yes |
| $W_K$ (`k_proj`) | Creates Key vectors | $(d, d)$ | Yes |
| $W_V$ (`v_proj`) | Creates Value vectors | $(d, d)$ | Yes |
| $W_O$ (`o_proj`) | Output projection | $(d, d)$ | Yes |
| $W_1, W_2$ (FFN) | Feed-forward network | $(d_{ff}, d), (d, d_{ff})$ | No |

That's 4 LoRA pairs per block × 28 blocks = 112 pairs of $A, B$
matrices for Qwen3-0.6B. The feed-forward layers are left frozen —
for classification tasks, attention-only LoRA is enough and keeps the
adapter small.

In [ ]:
# Total LoRA adapter size for pawpaw's default config

n_layers = 28
n_targets = 4  # q_proj, k_proj, v_proj, o_proj
d = 1024
rank = 16  # pawpaw default

params_per_adapter = 2 * rank * d  # A + B for one weight matrix
total_params = n_layers * n_targets * params_per_adapter
size_mb = total_params * 2 / (1024 ** 2)  # each param stored as 2-byte float

print(f"LoRA rank:      {rank}")
print(f"Adapters:       {n_layers} layers × {n_targets} projections = {n_layers * n_targets} pairs")
print(f"Params/pair:    {params_per_adapter:,}")
print(f"Total params:   {total_params:,}")
print(f"Disk size:      {size_mb:.1f} MB (stored as float16)")
print(f"")
print(f"Compare: the base model is ~600 MB.")
print(f"The adapter is {600 / size_mb:.0f}× smaller — you could fit ~86 different")
print(f"task-specific adapters in the space of one full model copy.")

### Initialization: why a fresh LoRA has zero effect

When pawpaw creates a new LoRA, it initializes the matrices as:

$$A \sim \mathcal{N}(0, \sigma^2) \qquad B = \mathbf{0}$$

($A$ gets small random values from a Gaussian, $B$ is set to all zeros.)
This means $B A = \mathbf{0}$ at the start of training — the correction
matrix is entirely zero, so the model behaves exactly as it did before
any training.

This is an important design choice: it guarantees that the starting
point is the pre-trained model, not something random. Training then
gradually nudges $A$ and $B$ away from their initial values, and the
LoRA correction "turns on" progressively.

The effective scale of the correction is $\alpha / r$. With pawpaw's
default $\alpha = 2r$, the scale is $2$. This means that by the end
of training, the LoRA correction has a total influence equivalent to
twice the magnitude of the random initialization of $A$ — enough to
reshape the model's behavior significantly, but still small enough
that it acts as a *correction* on top of $W$, not a replacement.

In [ ]:
import torch

torch.manual_seed(42)
d_in, d_out, r = 1024, 1024, 16

# The frozen original weight matrix W ∈ ℝ^(1024 × 1024)
W = torch.randn(d_out, d_in) * 0.02

# Fresh LoRA matrices — B = 0, so B×A = 0
A = torch.randn(r, d_in) * 0.02   # A ~ 𝒩(0, σ²)
B = torch.zeros(d_out, r)         # B = 0

# Compare: output with and without the fresh LoRA
x = torch.randn(d_in)
y_base = W @ x                    # W x  (original model)
y_lora = (W + B @ A) @ x          # (W + BA) x, but BA = 0

print(f"Output without LoRA:  {y_base[:3].tolist()}")
print(f"Output with LoRA:     {y_lora[:3].tolist()}")
print(f"Max difference:       {(y_base - y_lora).abs().max():.2e}")
print(f"\n→ A fresh LoRA is invisible. The model starts from its pre-trained state.")

# Now simulate a trained LoRA (B has been updated by gradient descent)
B_trained = torch.randn(d_out, r) * 0.01
y_trained = (W + B_trained @ A) @ x
delta = (B_trained @ A @ x).norm()
base_norm = (W @ x).norm()
print(f"\nAfter training:")
print(f"  ||W x||  = {base_norm:.2f}")
print(f"  ||BAx||  = {delta:.2f}")
print(f"  Ratio:   {delta / base_norm * 100:.1f}% of base model signal")
print(f"  → The correction is small but meaningful — a nudge, not a rewrite.")

### After training: what the LoRA learned

After training, $B A$ is no longer zero — it has become a correction
matrix that nudges the model's behavior toward the target task. But the
correction is **additive**: it sits on top of the original weights and
can be removed at any time by setting $B$ back to zero.

This has two important consequences:

1. **The original model is never modified.** You can train many
   different LoRA adapters for many different tasks, all sharing the
   same base model. That's the core insight that makes pawpaw's
   multi-program agent pattern possible.

2. **LoRA adapters can be swapped at runtime.** Changing which LoRA
   is active is just a pointer change in the inference engine — no
   weights need to be copied or reloaded. The next section explains
   this in detail.

There's also a useful mathematical property: if you want to permanently
apply the LoRA (no more swapping), you can **merge** it into the base
weights by computing $W' = W + (\alpha/r) B A$ and then discarding
$A$ and $B$. After merging, inference is exactly as fast as the
original model — zero overhead. pawpaw doesn't do this because it
needs the swap capability, but it's available for other use cases.

### How pawpaw trains the LoRA

1. **Synthesize training data**: A local LLM generates ~300 (input,
   output) pairs from the spec. For example, for a triage classifier,
   it creates examples like `("How are you?", "trivial")` and
   `("Debug my null pointer", "substantive")`. Alternatively, you
   provide your own `examples=`.

2. **Format as prompts**: Each example is rendered into a prompt
   template:
   ```
   [spec text + few-shot demos]
   Input: {user message}
   Output: {label}
   ```
   The spec and demos are the "prefix" — they tell the model what
   task it's performing. The prefix is the same for every example.

3. **Train**: Standard supervised fine-tuning, but only the LoRA
   $A$ and $B$ matrices receive gradient updates. The objective is
   the standard cross-entropy loss:

$$\mathcal{L} = -\sum_t \log P(\text{target}_t \mid \text{context}; W, A, B)$$

   Only $A$ and $B$ appear in the gradient; $W$ is frozen so
   $\nabla_W \mathcal{L} = 0$. This means the optimizer needs to
   track far fewer running averages (Adam has two buffers per
   parameter), which is why LoRA training uses ~3× less GPU memory
   than full fine-tuning.

4. **Save**: The frozen weights are *not* saved (they already
   exist in the base model GGUF). Only the ~7 MB of LoRA matrices
   are written to disk.

---

## 3. Quantization & GGUF

Modern LLMs store weights as float32 (4 bytes each) or float16 /
bfloat16 (2 bytes each). A 0.6B-parameter model in fp16 takes ~1.2 GB
in memory. That's too big for many laptops.

**Quantization** reduces the precision of the weights to save memory
and compute, with minimal quality loss.

### How quantization works

Instead of storing each weight as a 16-bit float, we group weights
into blocks (e.g. 32 or 256 values) and store:
- A **scale** factor per block (kept in fp16)
- The weight values as **small integers** (1–8 bits each)

For example, **Q6_K** (used by pawpaw for the base model) stores
each weight as approximately 6 bits instead of 16. This cuts the
model size by ~62% with negligible quality loss — the original
LoRA paper and many benchmarks confirm that 4–8 bit quantization
barely affects output quality for classification tasks.

In [ ]:
# Memory footprint comparison

params = 600_000_000  # 0.6B

formats = {
    "float32":  4.0,
    "float16":  2.0,
    "Q8_0":     1.0,   # ~8 bits = 1 byte per weight
    "Q6_K":     0.75,  # ~6 bits per weight
    "Q4_K_M":   0.56,  # ~4.5 bits per weight
    "Q4_0":     0.5,   # exactly 4 bits per weight
}

print(f"{'Format':>10} {'Bytes/param':>12} {'Model size':>12} {'VRAM saved':>12}")
print("-" * 52)
for name, bpw in formats.items():
    size_mb = params * bpw / (1024 ** 2)
    saved = (1 - bpw / 2.0) * 100  # vs fp16 baseline
    print(f"{name:>10} {bpw:>10.2f} B {size_mb:>8.0f} MB {saved:>10.0f}%")

### GGUF: the file format

[GGUF](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md) is a
binary format designed for **llama.cpp** — the C/C++ inference engine
that pawpaw uses under the hood (via the `llama-cpp-python` Python
binding).

A GGUF file contains:
- A **header** with metadata (architecture, vocabulary, context
  length, ...)
- **Tensor data** — the (possibly quantized) weights, ready for
  fast `mmap` loading

Key property: a GGUF file is **self-contained**. No external files
needed — copy it between machines and it just works.

GGUF also supports **LoRA adapter** files — separate, much smaller
GGUFs that contain only the $A$ and $B$ matrices.

### pawpaw uses two separate GGUF files

| File | Contents | Size | Quantization |
|---|---|---|---|
| Base model GGUF | All 600M weights | ~450 MB | Q6_K (6-bit) |
| Adapter GGUF | LoRA A & B matrices | ~5 MB | f16 (no quantization) |

**Why isn't the adapter quantized?** It's already tiny — quantizing
5 MB to 2.5 MB saves negligible space while introducing rounding
errors that can degrade classification accuracy. The base model
(450 MB → 1.2 GB without quantization) benefits enormously; the
adapter doesn't.

pawpaw downloads the base model GGUF once (from HuggingFace) and
caches it under `~/.cache/pawpaw/base_models/`. Multiple programs
share the same base model file.

---

## 4. The .paw bundle

When you call `pawpaw.compile(spec, save_to="programs/triage")`, pawpaw
writes two things:

1. **A bundle directory** (`programs/triage/`) with:
   - `adapter.gguf` — the LoRA adapter (f16, ~5 MB)
   - `prompt_template.txt` — the full prompt including spec + few-shot demos
   - `meta.json` — spec, base model, LoRA config, examples, versions
   - `prefix_kv_cache.bin` — precomputed KV cache for the prompt prefix

2. **A .paw file** (`programs/triage.paw`) — a single portable binary that
   contains all of the above packed into one file.

The `.paw` format is a simple binary container:
```
[4B magic] [4B version] [4B metadata_len] [metadata JSON] [safetensors blob]
```
It's binary-compatible with the 
[programasweights](https://programasweights.com) .paw format.

In [ ]:
# Let's peek inside a real .paw file (if one exists from the main tutorial)
from pathlib import Path
import struct, json

paw_paths = list(Path("programs").glob("**/*.paw")) if Path("programs").exists() else []

if paw_paths:
    p = paw_paths[0]
    with open(p, "rb") as f:
        magic = f.read(4)
        version = struct.unpack("<I", f.read(4))[0]
        meta_len = struct.unpack("<I", f.read(4))[0]
        metadata = json.loads(f.read(meta_len))
        tensor_bytes = len(f.read())

    print(f"File:         {p}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")
    print(f"Magic:        {magic!r}")
    print(f"Version:      {version}")
    print(f"Metadata:     {meta_len:,} bytes")
    print(f"Tensors:      {tensor_bytes:,} bytes ({tensor_bytes / 1024 / 1024:.1f} MB)")
    print(f"\nMetadata keys: {list(metadata.keys())}")
    print(f"Spec:         {metadata.get('spec', '')[:80]}...")
    print(f"Base model:   {metadata.get('interpreter_model', metadata.get('base_model', '?'))}")
    print(f"Has LoRA:     {metadata.get('has_lora', False)}")
    print(f"LoRA rank:    {metadata.get('lora_config', {}).get('rank', '?')}")
else:
    print("No .paw files found. Run the main tutorial.ipynb first to compile a program.")

---

## 5. Inference: loading & switching

This is where pawpaw's design pays off. Loading three classifiers costs
roughly the same memory as loading one. Here's why:

### The shared Llama context

`llama-cpp-python` creates a **Llama context** when you load a model.
This is the big allocation — it holds the base model weights and the
KV cache for the context window (~1 GB for Qwen3-0.6B at 4096 tokens).

pawpaw caches these contexts in a dict keyed by
`(base_model_path, n_ctx, n_gpu_layers)`. The second `pawpaw.load()`
call with the same base model **reuses the existing context** — no
extra copy.

### Hot-swapping LoRA adapters

Each `Program` holds a handle to its LoRA adapter (loaded via
`llama_adapter_lora_init`). Switching from one program to another is:

1. Call `llama_set_adapter_lora(ctx, adapter_B, scale=1.0)` — swaps
   which LoRA is active. This is **O(1)** — just a pointer change
   in the engine.
2. Restore the prefix KV cache for the new program — one disk read
   (~8 MB, or recompute from scratch if the cache doesn't exist yet).

No model weights are copied. No new GPU allocations.

In [ ]:
# Memory breakdown for 3 programs sharing one base model

base_model_mb = 450  # Q6_K GGUF
kv_cache_mb = 8 * 4  # ~8 MB per 1024 tokens of prefix, 4 programs
adapter_mb = 5  # per LoRA adapter
n_programs = 3

shared = base_model_mb + 32  # base model + llama overhead
per_program = adapter_mb + kv_cache_mb
total = shared + n_programs * per_program

naive = n_programs * (base_model_mb + adapter_mb + kv_cache_mb)

print(f"Base model (shared):       {shared:>6} MB")
print(f"Per program (adapter+KV):  {per_program:>6} MB")
print(f"3 programs, shared:        {total:>6} MB")
print(f"3 programs, naive:         {naive:>6} MB")
print(f"Savings:                   {naive - total:>6} MB ({(1 - total/naive)*100:.0f}%)")

### The prefix KV cache

Every call to a pawpaw program starts by evaluating the same prompt
prefix — the spec text, few-shot examples, and formatting instructions.
This prefix is typically 200–400 tokens and never changes.

After the first call, pawpaw saves the **KV cache** (the internal state
that the attention layers computed for the prefix) to
`prefix_kv_cache.bin` inside the bundle directory. On subsequent calls,
it restores this cache from disk instead of re-evaluating the prefix.

This is why the first call to a program takes ~1 second (cold start)
but every call after that takes ~50 ms (warm prefix + just the user
input).

When switching between programs, the prefix KV for the new program is
restored from its cache file. This is also why the bundle includes
`prefix_kv_cache.bin` — it's pre-computed so even the *very first*
call after `pawpaw.load()` is fast.

### The full inference flow

```
pawpaw.load("programs/triage")
  │
  ├─ Read bundle dir: meta.json, prompt_template.txt, adapter.gguf
  ├─ Get or create shared Llama context (download base model if needed)
  ├─ Load LoRA adapter handle via llama_adapter_lora_init()
  └─ Precompute prefix token IDs

triage("How are you?")
  │
  ├─ Activate: swap LoRA adapter + restore prefix KV from disk
  ├─ Tokenize user input ("How are you?") + suffix text
  ├─ Evaluate: feed tokens through the model (1 forward pass)
  ├─ Generate: autoregressive loop until EOS or max_tokens
  └─ Return: stripped output string ("trivial")
```

---

## Recap

| Concept | What it means for pawpaw |
|---|---|
| **Transformer** | The base model (Qwen3-0.6B) is a 28-layer transformer. Each layer has four attention weight matrices ($W_Q, W_K, W_V, W_O$) that mix context across tokens, plus a feed-forward network that refines each token individually. |
| **LoRA** | Instead of retraining all 600M params, pawpaw trains two tiny matrices per attention projection ($\Delta W = BA$, rank 16 → ~7 MB total). The original model stays frozen; the LoRA is an additive correction on top. |
| **Quantization** | The base model weights are stored as Q6_K (~6 bits per weight → ~450 MB instead of ~1.2 GB). The LoRA adapter stays in f16 (~5 MB) — too small to benefit from quantization. |
| **GGUF** | Self-contained binary format that llama.cpp loads directly. Separate GGUFs for base model and LoRA adapter. |
| **.paw file** | Single-file bundle with LoRA weights + metadata + prompt template. Binary-compatible with upstream programasweights. |
| **Shared context** | Multiple programs reuse one copy of the base model in memory. Switching programs is just a LoRA swap + KV cache restore — no weights copied. |
| **Prefix KV cache** | Pre-computed attention state for the prompt prefix. Saves ~200 ms per call after the first one. |

### Further reading

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — the original Transformer paper
- [LoRA: Low-Rank Adaptation](https://arxiv.org/abs/2106.09685) — the LoRA paper
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — visual walkthrough
- [HuggingFace PEFT: LoRA conceptual guide](https://huggingface.co/docs/peft/conceptual_guides/lora)
- [GGUF spec](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)
- [llama.cpp](https://github.com/ggerganov/llama.cpp) — the inference engine pawpaw uses